In [1]:
!pip install anthropic
import os
import anthropic
import ipywidgets as widgets
from IPython.display import display, clear_output

anthropic_key_input = widgets.Password(
    description='Anthropic Key:',
    placeholder='sk-ant-...',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)
spotify_id_input = widgets.Password(
    description='Spotify Client ID:',
    placeholder='your client id',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)
spotify_secret_input = widgets.Password(
    description='Spotify Secret:',
    placeholder='your client secret',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)

save_keys_btn = widgets.Button(
    description='✅ Save & Test Keys',
    button_style='success',
    layout=widgets.Layout(width='160px')
)
keys_output = widgets.Output()

def on_save_keys(b):
    with keys_output:
        clear_output()
        if anthropic_key_input.value:
            os.environ['ANTHROPIC_API_KEY'] = anthropic_key_input.value
        if spotify_id_input.value:
            os.environ['SPOTIFY_CLIENT_ID'] = spotify_id_input.value
        if spotify_secret_input.value:
            os.environ['SPOTIFY_CLIENT_SECRET'] = spotify_secret_input.value

        try:
            anthropic.Anthropic().models.list()
            print("✅ Anthropic key valid")
        except Exception as e:
            print(f"❌ Anthropic key failed: {e}")
            return

        try:
            spotify.authenticate()
            print("✅ Spotify credentials valid")
        except Exception as e:
            print(f"❌ Spotify credentials failed: {e}")
            return

        print("\n🎵 All keys saved! Scroll down to generate your playlist.")
        save_keys_btn.disabled = True
        for w in [anthropic_key_input, spotify_id_input, spotify_secret_input]:
            w.disabled = True

save_keys_btn.on_click(on_save_keys)

display(widgets.VBox([
    widgets.HTML("<h3>🔐 API Keys</h3><p style='color:gray;font-size:12px'>Keys are masked and only stored in memory for this session.</p>"),
    anthropic_key_input,
    spotify_id_input,
    spotify_secret_input,
    save_keys_btn,
    keys_output
]))

In [1]:
# =============================================================================
# 🎵 SPOTIFY PLAYLIST AGENT WITH ITERATIVE REFINEMENT
# =============================================================================
# A complete agentic system using Claude for creating personalized Spotify
# playlists with support for multi-turn refinement (perfect for weddings!)
# =============================================================================

# -----------------------------------------------------------------------------
# SECTION 1: Installation
# -----------------------------------------------------------------------------
import subprocess
import sys

def install_packages():
    packages = ['anthropic', 'httpx', 'python-dotenv']
    for package in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
    print("✅ Dependencies installed!")

install_packages()

# -----------------------------------------------------------------------------
# SECTION 2: Imports and Configuration
# -----------------------------------------------------------------------------
import os
import json
import base64
import httpx
from getpass import getpass
from typing import Optional, Any
from dataclasses import dataclass, field
from datetime import datetime
import anthropic

# Get API Keys securely
def setup_api_keys():
    if 'ANTHROPIC_API_KEY' not in os.environ:
        os.environ['ANTHROPIC_API_KEY'] = getpass('🔑 Enter your Anthropic API Key: ')

    if 'SPOTIFY_CLIENT_ID' not in os.environ:
        print("\n📝 To get Spotify credentials:")
        print("   1. Go to https://developer.spotify.com/dashboard")
        print("   2. Create an app (any name, select 'Web API')")
        print("   3. Copy Client ID and Client Secret\n")
        os.environ['SPOTIFY_CLIENT_ID'] = getpass('🎵 Enter Spotify Client ID: ')

    if 'SPOTIFY_CLIENT_SECRET' not in os.environ:
        os.environ['SPOTIFY_CLIENT_SECRET'] = getpass('🎵 Enter Spotify Client Secret: ')

    print("✅ API keys configured!")

setup_api_keys()

# -----------------------------------------------------------------------------
# SECTION 3: Spotify MCP Client
# -----------------------------------------------------------------------------
class SpotifyMCPClient:
    """
    Spotify client that simulates MCP server tools.
    Provides all the tools needed for playlist creation.
    """

    def __init__(self):
        self.client_id = os.environ['SPOTIFY_CLIENT_ID']
        self.client_secret = os.environ['SPOTIFY_CLIENT_SECRET']
        self.token_url = "https://accounts.spotify.com/api/token"
        self.api_base = "https://api.spotify.com/v1"
        self.access_token = None
        self._http_client = None

    @property
    def http_client(self):
        if self._http_client is None:
            self._http_client = httpx.Client(timeout=30.0)
        return self._http_client

    def authenticate(self):
        """Get Spotify access token using Client Credentials flow"""
        auth_string = f"{self.client_id}:{self.client_secret}"
        auth_bytes = base64.b64encode(auth_string.encode()).decode()

        response = self.http_client.post(
            self.token_url,
            headers={"Authorization": f"Basic {auth_bytes}"},
            data={"grant_type": "client_credentials"}
        )
        response.raise_for_status()
        self.access_token = response.json()["access_token"]
        return True

    def _get_headers(self):
        if not self.access_token:
            self.authenticate()
        return {"Authorization": f"Bearer {self.access_token}"}

    def search_tracks(self, query: str, limit: int = 20) -> list[dict]:
        """Search for tracks - MCP Tool #1"""
        try:
            response = self.http_client.get(
                f"{self.api_base}/search",
                headers=self._get_headers(),
                params={"q": query, "type": "track", "limit": min(limit, 50)}
            )
            response.raise_for_status()
            data = response.json()

            tracks = []
            for item in data.get("tracks", {}).get("items", []):
                tracks.append({
                    "id": item["id"],
                    "title": item["name"],
                    "artist": ", ".join(a["name"] for a in item["artists"]),
                    "artist_id": item["artists"][0]["id"] if item["artists"] else None,
                    "album": item["album"]["name"],
                    "duration_ms": item["duration_ms"],
                    "popularity": item["popularity"],
                    "spotify_url": item["external_urls"]["spotify"],
                    "preview_url": item.get("preview_url")
                })
            return tracks
        except Exception as e:
            return [{"error": str(e)}]

    def get_audio_features(self, track_ids: list[str]) -> list[dict]:
        """Get audio features (BPM, energy, danceability) - MCP Tool #2"""
        try:
            ids = ",".join(track_ids[:100])
            response = self.http_client.get(
                f"{self.api_base}/audio-features",
                headers=self._get_headers(),
                params={"ids": ids}
            )
            response.raise_for_status()
            data = response.json()

            features = []
            for item in data.get("audio_features", []):
                if item:
                    features.append({
                        "track_id": item["id"],
                        "bpm": round(item["tempo"]),
                        "energy": round(item["energy"], 2),
                        "danceability": round(item["danceability"], 2),
                        "valence": round(item["valence"], 2),
                        "acousticness": round(item["acousticness"], 2),
                        "instrumentalness": round(item["instrumentalness"], 2),
                        "loudness": round(item["loudness"], 1),
                        "speechiness": round(item["speechiness"], 2),
                        "key": item["key"],
                        "mode": "major" if item["mode"] == 1 else "minor",
                        "time_signature": item["time_signature"]
                    })
            return features
        except Exception as e:
            return [{"error": str(e)}]

    def search_artist(self, artist_name: str) -> dict:
        """Search for an artist - MCP Tool #3"""
        try:
            response = self.http_client.get(
                f"{self.api_base}/search",
                headers=self._get_headers(),
                params={"q": artist_name, "type": "artist", "limit": 1}
            )
            response.raise_for_status()
            data = response.json()

            artists = data.get("artists", {}).get("items", [])
            if artists:
                artist = artists[0]
                return {
                    "id": artist["id"],
                    "name": artist["name"],
                    "genres": artist.get("genres", []),
                    "popularity": artist["popularity"],
                    "followers": artist["followers"]["total"],
                    "spotify_url": artist["external_urls"]["spotify"],
                    "image_url": artist["images"][0]["url"] if artist.get("images") else None
                }
            return {"error": "Artist not found"}
        except Exception as e:
            return {"error": str(e)}

    def get_recommendations(
        self,
        seed_artists: list[str] = None,
        seed_genres: list[str] = None,
        seed_tracks: list[str] = None,
        target_bpm: int = None,
        min_bpm: int = None,
        max_bpm: int = None,
        target_energy: float = None,
        min_energy: float = None,
        max_energy: float = None,
        target_danceability: float = None,
        min_danceability: float = None,
        max_danceability: float = None,
        target_valence: float = None,
        min_valence: float = None,
        max_valence: float = None,
        target_acousticness: float = None,
        limit: int = 20
    ) -> list[dict]:
        """Get track recommendations - MCP Tool #4"""
        try:
            params = {"limit": min(limit, 100)}

            # Seeds (max 5 total)
            seeds_count = 0
            if seed_artists and seeds_count < 5:
                params["seed_artists"] = ",".join(seed_artists[:5-seeds_count])
                seeds_count += len(seed_artists[:5-seeds_count])
            if seed_genres and seeds_count < 5:
                params["seed_genres"] = ",".join(seed_genres[:5-seeds_count])
                seeds_count += len(seed_genres[:5-seeds_count])
            if seed_tracks and seeds_count < 5:
                params["seed_tracks"] = ",".join(seed_tracks[:5-seeds_count])

            # Target audio features
            if target_bpm: params["target_tempo"] = target_bpm
            if min_bpm: params["min_tempo"] = min_bpm
            if max_bpm: params["max_tempo"] = max_bpm
            if target_energy is not None: params["target_energy"] = target_energy
            if min_energy is not None: params["min_energy"] = min_energy
            if max_energy is not None: params["max_energy"] = max_energy
            if target_danceability is not None: params["target_danceability"] = target_danceability
            if min_danceability is not None: params["min_danceability"] = min_danceability
            if max_danceability is not None: params["max_danceability"] = max_danceability
            if target_valence is not None: params["target_valence"] = target_valence
            if min_valence is not None: params["min_valence"] = min_valence
            if max_valence is not None: params["max_valence"] = max_valence
            if target_acousticness is not None: params["target_acousticness"] = target_acousticness

            response = self.http_client.get(
                f"{self.api_base}/recommendations",
                headers=self._get_headers(),
                params=params
            )
            response.raise_for_status()
            data = response.json()

            tracks = []
            for item in data.get("tracks", []):
                tracks.append({
                    "id": item["id"],
                    "title": item["name"],
                    "artist": ", ".join(a["name"] for a in item["artists"]),
                    "artist_id": item["artists"][0]["id"] if item["artists"] else None,
                    "album": item["album"]["name"],
                    "duration_ms": item["duration_ms"],
                    "popularity": item["popularity"],
                    "spotify_url": item["external_urls"]["spotify"]
                })
            return tracks
        except Exception as e:
            return [{"error": str(e)}]

    def get_genre_seeds(self) -> list[str]:
        """Get available genre seeds - MCP Tool #5"""
        try:
            response = self.http_client.get(
                f"{self.api_base}/recommendations/available-genre-seeds",
                headers=self._get_headers()
            )
            response.raise_for_status()
            return response.json().get("genres", [])
        except Exception as e:
            return [f"error: {str(e)}"]

    def get_artist_top_tracks(self, artist_id: str, market: str = "US") -> list[dict]:
        """Get artist's top tracks - MCP Tool #6"""
        try:
            response = self.http_client.get(
                f"{self.api_base}/artists/{artist_id}/top-tracks",
                headers=self._get_headers(),
                params={"market": market}
            )
            response.raise_for_status()
            data = response.json()

            tracks = []
            for item in data.get("tracks", []):
                tracks.append({
                    "id": item["id"],
                    "title": item["name"],
                    "artist": ", ".join(a["name"] for a in item["artists"]),
                    "album": item["album"]["name"],
                    "popularity": item["popularity"],
                    "spotify_url": item["external_urls"]["spotify"]
                })
            return tracks
        except Exception as e:
            return [{"error": str(e)}]

# Test Spotify connection
spotify = SpotifyMCPClient()
try:
    spotify.authenticate()
    print("✅ Spotify connection successful!")
except Exception as e:
    print(f"❌ Spotify connection failed: {e}")

# -----------------------------------------------------------------------------
# SECTION 4: Playlist Session Data Classes
# -----------------------------------------------------------------------------
@dataclass
class PlaylistSession:
    """Maintains state for iterative playlist building"""
    session_id: str
    event_type: Optional[str] = None
    segments: dict = field(default_factory=dict)
    current_playlist: list = field(default_factory=list)
    user_preferences: dict = field(default_factory=dict)
    conversation_history: list = field(default_factory=list)
    refinement_count: int = 0
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

# -----------------------------------------------------------------------------
# SECTION 5: Iterative Playlist Agent
# -----------------------------------------------------------------------------
class IterativePlaylistAgent:
    """
    Agentic Spotify playlist creator with iterative refinement.
    Perfect for complex events like weddings with multiple segments.
    """

    def __init__(self, spotify_client: SpotifyMCPClient):
        self.client = anthropic.Anthropic()
        self.spotify = spotify_client
        self.model = "claude-sonnet-4-6"
        self.session: Optional[PlaylistSession] = None

        # Define tools (MCP-style interface)
        self.tools = [
            {
                "name": "search_tracks",
                "description": "Search Spotify for tracks by song title, artist name, lyrics, or keywords. Returns track IDs, titles, artists, popularity, and Spotify URLs.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "query": {
                            "type": "string",
                            "description": "Search query - can be song name, artist, or descriptive keywords"
                        },
                        "limit": {
                            "type": "integer",
                            "description": "Number of results (1-50)",
                            "default": 10
                        }
                    },
                    "required": ["query"]
                }
            },
            {
                "name": "get_audio_features",
                "description": "Get detailed audio features for tracks: BPM (tempo), energy (0-1), danceability (0-1), valence/happiness (0-1), acousticness (0-1). Essential for matching mood and energy requirements.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "track_ids": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "Spotify track IDs to analyze"
                        }
                    },
                    "required": ["track_ids"]
                }
            },
            {
                "name": "search_artist",
                "description": "Find an artist on Spotify and get their ID, genres, and popularity. Use this to get artist IDs for recommendations.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "artist_name": {
                            "type": "string",
                            "description": "Name of the artist to search"
                        }
                    },
                    "required": ["artist_name"]
                }
            },
            {
                "name": "get_recommendations",
                "description": "Get personalized track recommendations based on seed artists, genres, tracks, and target audio features. Max 5 seeds total across all seed types.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "seed_artists": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "Artist IDs to base recommendations on"
                        },
                        "seed_genres": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "Genres (use get_genre_seeds to see available options)"
                        },
                        "seed_tracks": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "Track IDs to base recommendations on"
                        },
                        "target_bpm": {
                            "type": "integer",
                            "description": "Target tempo in BPM"
                        },
                        "min_bpm": {"type": "integer"},
                        "max_bpm": {"type": "integer"},
                        "target_energy": {
                            "type": "number",
                            "description": "Target energy (0.0=calm to 1.0=intense)"
                        },
                        "min_energy": {"type": "number"},
                        "max_energy": {"type": "number"},
                        "target_danceability": {
                            "type": "number",
                            "description": "Target danceability (0.0 to 1.0)"
                        },
                        "target_valence": {
                            "type": "number",
                            "description": "Target happiness/positivity (0.0=sad to 1.0=happy)"
                        },
                        "target_acousticness": {
                            "type": "number",
                            "description": "Target acousticness (0.0=electronic to 1.0=acoustic)"
                        },
                        "limit": {
                            "type": "integer",
                            "default": 20
                        }
                    }
                }
            },
            {
                "name": "get_genre_seeds",
                "description": "Get list of all available genre seeds for recommendations. Use this to find valid genre names.",
                "input_schema": {
                    "type": "object",
                    "properties": {}
                }
            },
            {
                "name": "get_artist_top_tracks",
                "description": "Get an artist's most popular tracks. Useful for finding well-known songs by a specific artist.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "artist_id": {
                            "type": "string",
                            "description": "Spotify artist ID"
                        }
                    },
                    "required": ["artist_id"]
                }
            },
            {
                "name": "add_to_playlist",
                "description": "Add tracks to the current playlist with optional segment assignment (e.g., 'ceremony', 'cocktail_hour', 'first_dance', 'party')",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "tracks": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "id": {"type": "string"},
                                    "title": {"type": "string"},
                                    "artist": {"type": "string"},
                                    "spotify_url": {"type": "string"}
                                }
                            },
                            "description": "Tracks to add"
                        },
                        "segment": {
                            "type": "string",
                            "description": "Event segment (ceremony, cocktail_hour, dinner, first_dance, party, etc.)"
                        }
                    },
                    "required": ["tracks"]
                }
            },
            {
                "name": "remove_from_playlist",
                "description": "Remove specific tracks from the playlist by title or ID",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "track_identifiers": {
                            "type": "array",
                            "items": {"type": "string"},
                            "description": "Track IDs or titles to remove"
                        },
                        "segment": {
                            "type": "string",
                            "description": "Only remove from this segment (optional)"
                        }
                    },
                    "required": ["track_identifiers"]
                }
            },
            {
                "name": "get_current_playlist",
                "description": "Get the current playlist state with all segments and tracks",
                "input_schema": {
                    "type": "object",
                    "properties": {}
                }
            },
            {
                "name": "clear_segment",
                "description": "Clear all tracks from a specific segment",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "segment": {
                            "type": "string",
                            "description": "Segment to clear"
                        }
                    },
                    "required": ["segment"]
                }
            }
        ]

        self.system_prompt = """You are an expert music curator and DJ specializing in creating perfect playlists for events. You excel at understanding musical preferences and creating cohesive, well-flowing playlists.

## Your Expertise:
- **Wedding Events**: Ceremony music, cocktail hour, dinner, first dance, parent dances, party/reception
- **Parties**: Building energy, reading the room, genre mixing
- **Personal Playlists**: Workout, study, relaxation, road trips
- **Music Knowledge**: Deep understanding of genres, BPM, energy levels, and musical flow

## Audio Feature Guide:
- **BPM (Tempo)**: 60-80 (slow/ballads), 90-110 (moderate), 115-130 (upbeat), 130+ (high energy dance)
- **Energy** (0-1): 0.0-0.3 (calm/ambient), 0.4-0.6 (moderate), 0.7-1.0 (high energy)
- **Danceability** (0-1): 0.0-0.4 (not danceable), 0.5-0.7 (somewhat), 0.8-1.0 (dance floor ready)
- **Valence** (0-1): 0.0-0.3 (sad/melancholic), 0.4-0.6 (neutral), 0.7-1.0 (happy/euphoric)
- **Acousticness** (0-1): 0.0 (electronic/produced), 1.0 (acoustic/natural)

## Wedding Playlist Guidelines:
1. **Ceremony** (15-30 min): Processional, during ceremony, recessional. Mostly instrumental or soft vocals. Energy: 0.2-0.4
2. **Cocktail Hour** (45-60 min): Light jazz, acoustic covers, indie. Energy: 0.3-0.5
3. **Dinner** (60-90 min): Background music, conversations. Energy: 0.3-0.5
4. **First Dance**: Romantic, meaningful lyrics. Energy: 0.3-0.5
5. **Parent Dances**: Classic, sentimental songs
6. **Party/Reception**: Build energy progressively. Start 0.5, peak 0.8-0.9, end 0.6

## Iteration Approach:
- Present initial suggestions and ASK for feedback
- Be ready to swap songs, adjust energy, change genres
- Remember previous feedback in the session
- Offer alternatives when removing songs
- Explain WHY certain songs fit the vibe

## IMPORTANT WORKFLOW:
1. When searching for music, use the tools to find real tracks
2. After finding tracks, use add_to_playlist to save them to the session
3. Always organize tracks into appropriate segments
4. When user asks to remove songs, use remove_from_playlist
5. Use get_current_playlist to check what's been added

When creating playlists, always:
1. Confirm understanding of the request
2. Use tools to search and gather track data
3. Check audio features to ensure good fit
4. ADD tracks to the playlist using add_to_playlist tool
5. Present organized results with segments if applicable
6. Invite feedback for refinement"""

    def start_session(self, event_type: str = None) -> str:
        """Start a new playlist building session"""
        import uuid
        self.session = PlaylistSession(
            session_id=str(uuid.uuid4())[:8],
            event_type=event_type
        )
        return self.session.session_id

    def execute_tool(self, tool_name: str, tool_input: dict) -> str:
        """Execute a tool and return results"""
        try:
            if tool_name == "search_tracks":
                result = self.spotify.search_tracks(
                    tool_input["query"],
                    tool_input.get("limit", 10)
                )

            elif tool_name == "get_audio_features":
                result = self.spotify.get_audio_features(tool_input["track_ids"])

            elif tool_name == "search_artist":
                result = self.spotify.search_artist(tool_input["artist_name"])

            elif tool_name == "get_recommendations":
                result = self.spotify.get_recommendations(
                    seed_artists=tool_input.get("seed_artists"),
                    seed_genres=tool_input.get("seed_genres"),
                    seed_tracks=tool_input.get("seed_tracks"),
                    target_bpm=tool_input.get("target_bpm"),
                    min_bpm=tool_input.get("min_bpm"),
                    max_bpm=tool_input.get("max_bpm"),
                    target_energy=tool_input.get("target_energy"),
                    min_energy=tool_input.get("min_energy"),
                    max_energy=tool_input.get("max_energy"),
                    target_danceability=tool_input.get("target_danceability"),
                    target_valence=tool_input.get("target_valence"),
                    target_acousticness=tool_input.get("target_acousticness"),
                    limit=tool_input.get("limit", 20)
                )

            elif tool_name == "get_genre_seeds":
                result = self.spotify.get_genre_seeds()

            elif tool_name == "get_artist_top_tracks":
                result = self.spotify.get_artist_top_tracks(tool_input["artist_id"])

            elif tool_name == "add_to_playlist":
                segment = tool_input.get("segment", "general")
                if segment not in self.session.segments:
                    self.session.segments[segment] = []

                added_tracks = []
                for track in tool_input["tracks"]:
                    track_obj = {
                        "id": track.get("id"),
                        "title": track.get("title"),
                        "artist": track.get("artist"),
                        "spotify_url": track.get("spotify_url"),
                        "segment": segment
                    }
                    self.session.segments[segment].append(track_obj)
                    self.session.current_playlist.append(track_obj)
                    added_tracks.append(f"{track.get('title')} - {track.get('artist')}")

                result = {
                    "status": "success",
                    "added": len(tool_input["tracks"]),
                    "segment": segment,
                    "tracks_added": added_tracks,
                    "total_in_segment": len(self.session.segments[segment])
                }

            elif tool_name == "remove_from_playlist":
                identifiers = [i.lower() for i in tool_input["track_identifiers"]]
                segment = tool_input.get("segment")
                removed = []

                if segment and segment in self.session.segments:
                    original = self.session.segments[segment]
                    self.session.segments[segment] = [
                        t for t in original
                        if t["id"].lower() not in identifiers
                        and t["title"].lower() not in identifiers
                    ]
                    removed = [t for t in original if t not in self.session.segments[segment]]
                else:
                    for seg in list(self.session.segments.keys()):
                        original = self.session.segments[seg]
                        self.session.segments[seg] = [
                            t for t in original
                            if t["id"].lower() not in identifiers
                            and t["title"].lower() not in identifiers
                        ]
                        removed.extend([t for t in original if t not in self.session.segments[seg]])

                self.session.current_playlist = []
                for seg in self.session.segments:
                    self.session.current_playlist.extend(self.session.segments[seg])

                result = {
                    "status": "success",
                    "removed": len(removed),
                    "removed_tracks": [f"{t['title']} - {t['artist']}" for t in removed]
                }

            elif tool_name == "get_current_playlist":
                result = {
                    "session_id": self.session.session_id,
                    "event_type": self.session.event_type,
                    "total_tracks": len(self.session.current_playlist),
                    "segments": {
                        seg: {
                            "track_count": len(tracks),
                            "tracks": [{"title": t["title"], "artist": t["artist"], "spotify_url": t.get("spotify_url")} for t in tracks]
                        }
                        for seg, tracks in self.session.segments.items()
                    },
                    "refinement_count": self.session.refinement_count
                }

            elif tool_name == "clear_segment":
                segment = tool_input["segment"]
                if segment in self.session.segments:
                    cleared = len(self.session.segments[segment])
                    self.session.segments[segment] = []
                    self.session.current_playlist = [
                        t for t in self.session.current_playlist if t["segment"] != segment
                    ]
                    result = {"status": "success", "cleared": cleared, "segment": segment}
                else:
                    result = {"status": "error", "message": f"Segment '{segment}' not found"}
            else:
                result = {"error": f"Unknown tool: {tool_name}"}

            return json.dumps(result, indent=2)

        except Exception as e:
            return json.dumps({"error": str(e)})

    def chat(self, user_message: str, verbose: bool = True) -> str:
        """
        Process a user message and return agent response.
        Supports iterative refinement through conversation.
        """
        if self.session is None:
            self.start_session()

        self.session.conversation_history.append({
            "role": "user",
            "content": user_message
        })

        messages = self.session.conversation_history.copy()

        while True:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=4096,
                system=self.system_prompt,
                tools=self.tools,
                messages=messages
            )

            if response.stop_reason == "tool_use":
                assistant_content = response.content
                messages.append({"role": "assistant", "content": assistant_content})

                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        if verbose:
                            print(f"  🔧 {block.name}")
                            if block.name not in ["get_genre_seeds", "get_current_playlist"]:
                                input_str = json.dumps(block.input)
                                if len(input_str) > 100:
                                    input_str = input_str[:100] + "..."
                                print(f"     └─ {input_str}")

                        result = self.execute_tool(block.name, block.input)

                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result
                        })

                messages.append({"role": "user", "content": tool_results})

            elif response.stop_reason == "end_turn":
                final_text = ""
                for block in response.content:
                    if hasattr(block, "text"):
                        final_text += block.text

                self.session.conversation_history.append({
                    "role": "assistant",
                    "content": final_text
                })
                self.session.refinement_count += 1

                return final_text

            else:
                return f"Unexpected stop reason: {response.stop_reason}"

    def get_playlist_summary(self) -> dict:
        """Get a summary of the current playlist"""
        if not self.session:
            return {"error": "No active session"}

        summary = {
            "session_id": self.session.session_id,
            "event_type": self.session.event_type,
            "total_tracks": len(self.session.current_playlist),
            "segments": {},
            "refinements": self.session.refinement_count
        }

        for segment, tracks in self.session.segments.items():
            summary["segments"][segment] = {
                "count": len(tracks),
                "tracks": [f"{t['title']} - {t['artist']}" for t in tracks]
            }

        return summary

    def export_playlist(self, format: str = "text") -> str:
        """Export the current playlist"""
        if not self.session:
            return "No active session"

        if format == "text":
            output = f"# Playlist - {self.session.event_type or 'Custom'}\n"
            output += f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n"
            output += "=" * 50 + "\n\n"

            for segment, tracks in self.session.segments.items():
                if tracks:
                    output += f"## {segment.replace('_', ' ').title()}\n"
                    for i, track in enumerate(tracks, 1):
                        output += f"  {i}. {track['title']} - {track['artist']}\n"
                        if track.get('spotify_url'):
                            output += f"     🔗 {track['spotify_url']}\n"
                    output += "\n"

            return output

        elif format == "json":
            return json.dumps(self.session.segments, indent=2)

        elif format == "urls":
            urls = []
            for tracks in self.session.segments.values():
                for track in tracks:
                    if track.get('spotify_url'):
                        urls.append(track['spotify_url'])
            return "\n".join(urls)

        return "Unknown format"

# Initialize the agent
agent = IterativePlaylistAgent(spotify)
print("✅ Playlist Agent initialized!")

# -----------------------------------------------------------------------------
# SECTION 6: Interactive Chat Function
# -----------------------------------------------------------------------------
def chat_with_agent():
    """Interactive chat loop for playlist creation"""
    print("\n" + "=" * 70)
    print("🎵 SPOTIFY PLAYLIST AGENT - Interactive Mode")
    print("=" * 70)
    print("\n📝 Commands:")
    print("   'new <type>'  - Start new session (wedding, party, workout, study, custom)")
    print("   'export'      - Export current playlist")
    print("   'summary'     - Show playlist summary")
    print("   'quit'        - Exit")
    print("\n💡 Tips:")
    print("   - Describe artists, genres, mood, energy level, BPM preferences")
    print("   - For weddings, mention segments: ceremony, cocktail, dinner, party")
    print("   - Ask to add, remove, or swap songs")
    print("   - Request changes like 'more upbeat' or 'remove explicit songs'")
    print("-" * 70)

    agent.start_session("custom")
    print("\n✨ New session started! Describe your ideal playlist.\n")

    while True:
        try:
            user_input = input("\n🎤 You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\nGoodbye! 🎶")
            break

        if not user_input:
            continue

        lower_input = user_input.lower()

        if lower_input == 'quit':
            print("\nThanks for using Spotify Playlist Agent! 🎶")
            break

        if lower_input.startswith('new'):
            parts = lower_input.split()
            event_type = parts[1] if len(parts) > 1 else "custom"
            agent.start_session(event_type)
            print(f"\n✨ New {event_type} session started!")
            continue

        if lower_input == 'export':
            print("\n" + agent.export_playlist("text"))
            continue

        if lower_input == 'summary':
            summary = agent.get_playlist_summary()
            print(f"\n📊 Playlist Summary:")
            print(f"   Session: {summary['session_id']}")
            print(f"   Type: {summary['event_type']}")
            print(f"   Total Tracks: {summary['total_tracks']}")
            print(f"   Iterations: {summary['refinements']}")
            if summary.get('segments'):
                print(f"\n   Segments:")
                for seg, data in summary['segments'].items():
                    print(f"   • {seg}: {data['count']} tracks")
            continue

        print("\n🤖 Working on your request...\n")
        response = agent.chat(user_input)
        print(f"\n🎵 Agent:\n{response}")

# -----------------------------------------------------------------------------
# SECTION 7: Demo Function - Wedding Playlist
# -----------------------------------------------------------------------------
def run_wedding_demo():
    """Demonstrate wedding playlist creation with iterations"""
    print("\n" + "=" * 70)
    print("🎵 WEDDING PLAYLIST DEMO")
    print("=" * 70)

    agent.start_session("wedding")

    # Initial request
    print("\n📝 Step 1: Initial Wedding Request")
    print("-" * 50)
    response = agent.chat("""
    I'm planning music for my wedding! We need:

    1. CEREMONY: Romantic instrumental for walking down the aisle
    2. FIRST DANCE: We love Ed Sheeran - something romantic
    3. PARTY: Mix of current hits and 80s classics, high energy for dancing

    We want a romantic but fun vibe overall!
    """)
    print(f"\n🎵 Agent:\n{response}")

    # First iteration
    print("\n" + "=" * 70)
    print("📝 Step 2: Refine the First Dance")
    print("-" * 50)
    response = agent.chat("""
    The first dance options look great! But can you also check the
    BPM and energy of "Perfect" by Ed Sheeran? And maybe suggest
    one alternative that's slightly more upbeat?
    """)
    print(f"\n🎵 Agent:\n{response}")

    # Second iteration
    print("\n" + "=" * 70)
    print("📝 Step 3: Adjust Party Music")
    print("-" * 50)
    response = agent.chat("""
    For the party section, can you add "September" by Earth Wind & Fire?
    Also, we want songs that everyone can dance to - around 120 BPM.
    """)
    print(f"\n🎵 Agent:\n{response}")

    # Export final playlist
    print("\n" + "=" * 70)
    print("📋 FINAL PLAYLIST EXPORT")
    print("=" * 70)
    print(agent.export_playlist("text"))

    # Summary
    summary = agent.get_playlist_summary()
    print(f"\n📊 Session Stats:")
    print(f"   Total Tracks: {summary['total_tracks']}")
    print(f"   Iterations: {summary['refinements']}")

# -----------------------------------------------------------------------------
# SECTION 8: Quick Demo Function
# -----------------------------------------------------------------------------
def run_quick_demo():
    """Run a quick single-request demo"""
    print("\n" + "=" * 70)
    print("🏃 QUICK DEMO: Workout Playlist")
    print("=" * 70)

    agent.start_session("workout")

    response = agent.chat("""
    Create a 20-minute high-energy workout playlist:
    - BPM between 130-150
    - High energy (0.8+)
    - Mix of electronic and pop
    - Artists like Calvin Harris, Dua Lipa, The Weeknd
    """)

    print(f"\n🎵 Agent:\n{response}")

    print("\n📋 Playlist Export:")
    print(agent.export_playlist("text"))

# -----------------------------------------------------------------------------
# SECTION 9: Main Execution
# -----------------------------------------------------------------------------
print("\n" + "=" * 70)
print("🎵 SPOTIFY PLAYLIST AGENT READY!")
print("=" * 70)
print("\n📌 Available Functions:")
print("   • chat_with_agent()   - Interactive chat mode")
print("   • run_wedding_demo()  - See wedding playlist creation")
print("   • run_quick_demo()    - Quick workout playlist demo")
print("\n📌 Direct Usage:")
print("   • agent.chat('your request')  - Single message")
print("   • agent.export_playlist()     - Export current playlist")
print("   • agent.get_playlist_summary() - Get summary")
print("\n" + "=" * 70)

# Uncomment one of these to run:
# chat_with_agent()      # Interactive mode
# run_wedding_demo()     # Wedding demo
# run_quick_demo()       # Quick demo

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# DISCOGS WORKFLOW CELL
# Add this cell AFTER the agent is initialized and BEFORE the last widget cell.
# =============================================================================
# Workflow:
#   1. Enter Discogs token + username → fetch full collection (albums CSV)
#   2. Send release IDs to discogs-7dh1.onrender.com → expand to tracks CSV
#   3. Match each track to Spotify → build a Spotify-ready track list
#   4. Load matched tracks into the playlist agent for generation/refinement
# =============================================================================

import os
import time
import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# ---------------------------------------------------------------------------
# Secure Discogs key input
# ---------------------------------------------------------------------------
discogs_token_input = widgets.Password(
    description='Discogs Token:',
    placeholder='your personal access token',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)
discogs_username_input = widgets.Text(
    description='Discogs Username:',
    placeholder='your discogs username',
    style={'description_width': '130px'},
    layout=widgets.Layout(width='500px')
)
save_discogs_btn = widgets.Button(
    description='✅ Save & Test',
    button_style='success',
    layout=widgets.Layout(width='130px')
)
discogs_key_output = widgets.Output()

def on_save_discogs(b):
    with discogs_key_output:
        clear_output()
        token = discogs_token_input.value.strip()
        username = discogs_username_input.value.strip()
        if not token or not username:
            print("⚠️ Please enter both your token and username.")
            return
        os.environ['DISCOGS_TOKEN'] = token
        os.environ['DISCOGS_USERNAME'] = username
        # Quick identity check
        r = requests.get(
            'https://api.discogs.com/oauth/identity',
            headers={
                'Authorization': f'Discogs token={token}',
                'User-Agent': 'PlaylistAgent/1.0'
            }
        )
        if r.status_code == 200:
            data = r.json()
            print(f"✅ Connected as: {data.get('username')} — ready to fetch collection!")
            save_discogs_btn.disabled = True
            discogs_token_input.disabled = True
            discogs_username_input.disabled = True
        else:
            print(f"❌ Auth failed (HTTP {r.status_code}). Check your token and try again.")

save_discogs_btn.on_click(on_save_discogs)

display(widgets.VBox([
    widgets.HTML("<h3>🎛️ Discogs Setup</h3>"
                 "<p style='color:gray;font-size:12px'>Token is masked and only stored in memory.</p>"),
    discogs_token_input,
    discogs_username_input,
    save_discogs_btn,
    discogs_key_output,
]))


# ---------------------------------------------------------------------------
# Step 1: Fetch Discogs Collection  →  albums DataFrame
# ---------------------------------------------------------------------------
DISCOGS_API = 'https://api.discogs.com'

def fetch_discogs_collection(username: str, token: str) -> pd.DataFrame:
    """
    Page through the user's Discogs collection (folder 0 = all releases).
    Returns a DataFrame with one row per album.
    """
    headers = {
        'Authorization': f'Discogs token={token}',
        'User-Agent': 'PlaylistAgent/1.0'
    }
    releases = []
    page = 1
    total_pages = 1  # updated after first request

    print(f"📦 Fetching collection for '{username}'...")
    while page <= total_pages:
        r = requests.get(
            f'{DISCOGS_API}/users/{username}/collection/folders/0/releases',
            headers=headers,
            params={'page': page, 'per_page': 100, 'sort': 'artist', 'sort_order': 'asc'}
        )
        r.raise_for_status()
        data = r.json()
        pagination = data.get('pagination', {})
        total_pages = pagination.get('pages', 1)
        print(f"  Page {page}/{total_pages}  ({pagination.get('items', '?')} total albums)")

        for item in data.get('releases', []):
            info = item.get('basic_information', {})
            artists = info.get('artists', [])
            labels  = info.get('labels', [])
            formats = info.get('formats', [])
            releases.append({
                'release_id': info.get('id'),
                'artist':     ', '.join(a['name'] for a in artists),
                'title':      info.get('title'),
                'year':       info.get('year'),
                'label':      labels[0]['name'] if labels else '',
                'catno':      labels[0].get('catno', '') if labels else '',
                'format':     formats[0]['name'] if formats else '',
                'rating':     item.get('rating', 0),
                'discogs_url': f"https://www.discogs.com/release/{info.get('id')}",
            })

        page += 1
        time.sleep(0.5)  # respect Discogs rate limit (60 req/min)

    df = pd.DataFrame(releases)
    print(f"\n✅ Collection fetched: {len(df)} albums")
    return df


# ---------------------------------------------------------------------------
# Step 2: Expand albums → tracks via discogs-7dh1.onrender.com
# ---------------------------------------------------------------------------
DISCOGS_EXPLORER = 'https://discogs-7dh1.onrender.com'

def fetch_tracks_for_releases(release_ids: list[int], batch_size: int = 50) -> pd.DataFrame:
    """
    Send release IDs to the Discogs Explorer multi-release endpoint in batches.
    Returns a DataFrame with one row per track.
    """
    all_tracks = []
    batches = [release_ids[i:i+batch_size] for i in range(0, len(release_ids), batch_size)]

    for i, batch in enumerate(batches):
        print(f"  Fetching tracks batch {i+1}/{len(batches)} ({len(batch)} releases)...")
        try:
            r = requests.post(
                f'{DISCOGS_EXPLORER}/multi-release-csv',
                json={'release_ids': batch},
                timeout=120
            )
            if r.status_code == 200:
                # Parse CSV response
                import io
                batch_df = pd.read_csv(io.StringIO(r.text))
                all_tracks.append(batch_df)
            else:
                # Fallback: hit the GET endpoint per release
                print(f"    ⚠️ Batch endpoint returned {r.status_code}, falling back to per-release...")
                for rid in batch:
                    try:
                        r2 = requests.get(
                            f'{DISCOGS_EXPLORER}/release-csv',
                            params={'release_id': rid},
                            timeout=30
                        )
                        if r2.status_code == 200:
                            import io
                            df2 = pd.read_csv(io.StringIO(r2.text))
                            all_tracks.append(df2)
                        time.sleep(0.3)
                    except Exception as e:
                        print(f"    ⚠️ Skipped release {rid}: {e}")
        except Exception as e:
            print(f"  ❌ Batch {i+1} failed: {e}")
        time.sleep(1)

    if not all_tracks:
        return pd.DataFrame()

    tracks_df = pd.concat(all_tracks, ignore_index=True)
    # Normalize column names
    tracks_df.columns = [c.strip().lower().replace(' ', '_') for c in tracks_df.columns]
    print(f"\n✅ Tracks fetched: {len(tracks_df)} total tracks across {len(release_ids)} albums")
    return tracks_df


# ---------------------------------------------------------------------------
# Step 3: Match tracks to Spotify
# ---------------------------------------------------------------------------
def match_tracks_to_spotify(tracks_df: pd.DataFrame,
                             max_tracks: int = 500,
                             sample: bool = False) -> pd.DataFrame:
    """
    For each track, search Spotify and grab the best match.
    Returns the tracks_df with added Spotify columns.
    """
    # Identify the right columns (flexible naming)
    title_col  = next((c for c in tracks_df.columns if 'title' in c and 'release' not in c), None)
    artist_col = next((c for c in tracks_df.columns if 'artist' in c), None)

    if not title_col or not artist_col:
        print(f"⚠️ Could not find title/artist columns. Available: {list(tracks_df.columns)}")
        return tracks_df

    df = tracks_df.copy()
    if sample and len(df) > max_tracks:
        print(f"ℹ️ Sampling {max_tracks} tracks from {len(df)} for speed. Set sample=False for full match.")
        df = df.sample(max_tracks, random_state=42).reset_index(drop=True)
    elif len(df) > max_tracks:
        print(f"ℹ️ Limiting to first {max_tracks} tracks. Increase max_tracks if needed.")
        df = df.head(max_tracks).reset_index(drop=True)

    spotify_ids    = []
    spotify_urls   = []
    spotify_titles = []
    matched        = 0

    print(f"🔍 Matching {len(df)} tracks to Spotify...")
    for i, row in df.iterrows():
        track_name  = str(row.get(title_col, '')).strip()
        artist_name = str(row.get(artist_col, '')).strip()

        if not track_name or track_name.lower() in ('nan', ''):
            spotify_ids.append(None)
            spotify_urls.append(None)
            spotify_titles.append(None)
            continue

        query = f"{track_name} {artist_name}"
        results = spotify.search_tracks(query, limit=1)

        if results and 'error' not in results[0] and results[0].get('id'):
            r = results[0]
            spotify_ids.append(r['id'])
            spotify_urls.append(r['spotify_url'])
            spotify_titles.append(r['title'])
            matched += 1
        else:
            spotify_ids.append(None)
            spotify_urls.append(None)
            spotify_titles.append(None)

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(df)} processed  ({matched} matched so far)...")
        time.sleep(0.1)  # gentle rate limiting

    df['spotify_id']    = spotify_ids
    df['spotify_url']   = spotify_urls
    df['spotify_title'] = spotify_titles
    df['matched']       = df['spotify_id'].notna()

    print(f"\n✅ Spotify matching complete: {matched}/{len(df)} tracks matched "
          f"({100*matched//len(df)}%)")
    return df


# ---------------------------------------------------------------------------
# Step 4: Load matched tracks into the playlist agent
# ---------------------------------------------------------------------------
def load_collection_into_agent(matched_df: pd.DataFrame,
                                segment: str = 'collection',
                                limit: int = 200):
    """
    Takes matched tracks and loads them into the agent's current session
    so you can refine and generate playlists from your collection.
    """
    matched = matched_df[matched_df['matched'] == True].copy()

    if matched.empty:
        print("⚠️ No matched tracks to load.")
        return

    if len(matched) > limit:
        print(f"ℹ️ Loading a random sample of {limit} from {len(matched)} matched tracks.")
        matched = matched.sample(limit, random_state=42)

    title_col  = next((c for c in matched.columns if 'title' in c and 'spotify' not in c and 'release' not in c), 'track_title')
    artist_col = next((c for c in matched.columns if 'artist' in c), 'artist')

    tracks_to_add = []
    for _, row in matched.iterrows():
        tracks_to_add.append({
            'id':          row['spotify_id'],
            'title':       row.get(title_col, ''),
            'artist':      row.get(artist_col, ''),
            'spotify_url': row.get('spotify_url', '')
        })

    agent.start_session('collection')
    agent.execute_tool('add_to_playlist', {
        'tracks': tracks_to_add,
        'segment': segment
    })
    print(f"✅ {len(tracks_to_add)} tracks loaded into agent session '{segment}'.")
    print("💡 Now head to the playlist widget below and refine your collection into a playlist!")


# ---------------------------------------------------------------------------
# Main Widget UI
# ---------------------------------------------------------------------------
run_btn       = widgets.Button(description='🚀 Run Full Workflow', button_style='warning',
                                layout=widgets.Layout(width='200px'))
step_output   = widgets.Output()

# Storage for results across steps
_state = {
    'collection_df': None,
    'tracks_df':     None,
    'matched_df':    None,
}

def on_run_workflow(b):
    token    = os.environ.get('DISCOGS_TOKEN', '').strip()
    username = os.environ.get('DISCOGS_USERNAME', '').strip()

    if not token or not username:
        with step_output:
            clear_output()
            print("⚠️ Please save your Discogs token and username above first.")
        return

    with step_output:
        clear_output()

        # Step 1: Collection
        print("=" * 60)
        print("STEP 1: Fetching Discogs Collection")
        print("=" * 60)
        try:
            collection_df = fetch_discogs_collection(username, token)
            _state['collection_df'] = collection_df
            display(collection_df.head(10))
            print(f"... and {max(0, len(collection_df)-10)} more albums.\n")
        except Exception as e:
            print(f"❌ Collection fetch failed: {e}")
            return

        # Step 2: Tracks
        print("=" * 60)
        print("STEP 2: Expanding Albums → Tracks")
        print("=" * 60)
        release_ids = collection_df['release_id'].dropna().astype(int).tolist()
        try:
            tracks_df = fetch_tracks_for_releases(release_ids)
            _state['tracks_df'] = tracks_df
            if tracks_df.empty:
                print("❌ No tracks returned. Check the Explorer service.")
                return
            display(tracks_df.head(10))
            print(f"... and {max(0, len(tracks_df)-10)} more tracks.\n")
        except Exception as e:
            print(f"❌ Track expansion failed: {e}")
            return

        # Step 3: Spotify matching
        print("=" * 60)
        print("STEP 3: Matching Tracks to Spotify")
        print("=" * 60)
        try:
            matched_df = match_tracks_to_spotify(tracks_df, max_tracks=300, sample=False)
            _state['matched_df'] = matched_df
            matched_count = matched_df['matched'].sum()
            display(matched_df[matched_df['matched']].head(10))
            print(f"\n📊 {matched_count} Spotify-ready tracks from your collection.\n")
        except Exception as e:
            print(f"❌ Spotify matching failed: {e}")
            return

        # Step 4: Load into agent
        print("=" * 60)
        print("STEP 4: Loading into Playlist Agent")
        print("=" * 60)
        load_collection_into_agent(_state['matched_df'])

        print("\n🎉 Done! Scroll down to the Playlist Generator.")
        print("Your collection is loaded — describe the vibe you want and hit Generate.")

run_btn.on_click(on_run_workflow)

display(widgets.VBox([
    widgets.HTML("<hr><h3>🎛️ Discogs → Spotify Workflow</h3>"
                 "<p style='color:gray;font-size:12px'>"
                 "Fetches your full Discogs collection, expands to tracks, "
                 "matches to Spotify, and loads into the playlist agent.</p>"),
    run_btn,
    step_output,
]))

# Convenience: access results directly after running
# _state['collection_df']  →  albums
# _state['tracks_df']      →  all tracks
# _state['matched_df']     →  tracks with Spotify IDs

In [ ]:
# =============================================================================
# LAST CELL: Playlist Widget
# =============================================================================

import pandas as pd
import json
import re
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Global state ---
current_playlist_df = None
playlist_output     = widgets.Output()
status_output       = widgets.Output()

# --- Helper: get related artists ---
def get_related_artists(artist_id: str, limit: int = 5) -> list[dict]:
    try:
        response = spotify.http_client.get(
            f"{spotify.api_base}/artists/{artist_id}/related-artists",
            headers=spotify._get_headers()
        )
        response.raise_for_status()
        artists = response.json().get("artists", [])
        return [{"id": a["id"], "name": a["name"], "popularity": a["popularity"]}
                for a in artists[:limit]]
    except Exception:
        return []

def get_similar_artist_names(artist_name: str, n: int = 3) -> list[str]:
    artist_info = spotify.search_artist(artist_name)
    if "error" in artist_info:
        return []
    related = get_related_artists(artist_info["id"], limit=n)
    return [a["name"] for a in related]

# --- Input Fields ---
genre     = widgets.Text(placeholder='e.g. lo-fi, hip hop, jazz',               description='Genre:',    style={'description_width': '80px'})
vibe      = widgets.Text(placeholder='e.g. chill, hype, romantic',              description='Vibe:',     style={'description_width': '80px'})
venue     = widgets.Text(placeholder='e.g. wedding, gym, coffee shop',          description='Venue:',    style={'description_width': '80px'})
artist    = widgets.Text(placeholder='e.g. Drake, Tame Impala, Billie Eilish',  description='Artist:',   style={'description_width': '80px'})
song      = widgets.Text(placeholder='e.g. Bohemian Rhapsody, Blinding Lights', description='Song:',     style={'description_width': '80px'})
album     = widgets.Text(placeholder='e.g. Currents, After Hours',              description='Album:',    style={'description_width': '80px'})
year      = widgets.Text(placeholder='e.g. 1975, 1965-1975, 1990s',             description='Year/Era:', style={'description_width': '80px'})

harmonic_key = widgets.Dropdown(
    options=['N/A - Premium API Required'],
    description='Key:',
    disabled=True,
    style={'description_width': '80px'}
)
key_label = widgets.HTML("<p style='color:gray;font-size:11px;margin:0'>⚠️ Requires Spotify Premium API</p>")

bpm_min   = widgets.IntSlider(value=60,  min=40, max=200, description='BPM Min:', style={'description_width': '80px'})
bpm_max   = widgets.IntSlider(value=120, min=40, max=200, description='BPM Max:', style={'description_width': '80px'})
energy    = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1, description='Energy:',  style={'description_width': '80px'})
valence   = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1, description='Valence:', style={'description_width': '80px'})
num_songs = widgets.IntSlider(value=10, min=5, max=50, description='# Songs:', style={'description_width': '80px'})

popularity_range = widgets.IntRangeSlider(
    value=[50, 100], min=0, max=100, step=5,
    description='Popularity:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='400px'),
    tooltip='0 = deep cuts / obscure, 100 = most popular'
)
popularity_label = widgets.HTML(
    "<p style='color:gray;font-size:11px;margin:0'>0 = deep cuts &nbsp;|&nbsp; 100 = chart-toppers</p>"
)

generate_btn = widgets.Button(description='🎵 Generate Playlist', button_style='success')

# --- Refinement row (hidden until playlist is generated) ---
refinement_input = widgets.Text(
    placeholder='e.g. add Ripple by Grateful Dead, make it more upbeat',
    description='Refine:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='70%')
)
refine_btn = widgets.Button(description='✏️ Refine',   button_style='info')
done_btn   = widgets.Button(description='✅ Finalize', button_style='success')

refinement_row = widgets.VBox([
    widgets.HTML("<hr><b>💬 Refine your playlist:</b>"),
    widgets.HTML("<p style='color:gray;font-size:12px'>e.g. 'add Song X by Artist', 'replace track 3', 'more upbeat'</p>"),
    widgets.HBox([refinement_input, refine_btn, done_btn]),
], layout=widgets.Layout(display='none'))  # hidden by default

finalized_label = widgets.HTML("")

# --- Prompt builder ---
def build_prompt(num: int) -> str:
    half = max(1, num // 2)
    other_half = num - half

    pop_min, pop_max = popularity_range.value
    if pop_max >= 80:
        pop_description = "well-known, popular tracks"
    elif pop_max >= 50:
        pop_description = "moderately well-known tracks, a mix of hits and album cuts"
    else:
        pop_description = "obscure deep cuts, B-sides, and lesser-known tracks"
    if pop_min <= 20 and pop_max >= 80:
        pop_description = "a full range from deep cuts to chart-toppers"

    parts = [
        f"Create a {num}-song playlist.",
        "Return ONLY a JSON array, no other text, no markdown. Like this:",
        '[{"title":"Song","artist":"Artist","album":"Album","genre":"Genre","vibe":"Vibe","year":"1975","bpm":90,"energy":0.5,"valence":0.5,"why_it_fits":"reason"}]',
    ]

    if genre.value:  parts.append(f"Genre: {genre.value}")
    if vibe.value:   parts.append(f"Vibe: {vibe.value}")
    if venue.value:  parts.append(f"Venue: {venue.value}")
    if album.value:  parts.append(f"Album: {album.value}")
    if year.value:   parts.append(f"Year/Era: {year.value}")
    if song.value:   parts.append(f"Must include these specific songs: {song.value}.")

    parts.append(f"BPM: {bpm_min.value}-{bpm_max.value}, Energy: {energy.value}, Valence: {valence.value}")
    parts.append(f"Popularity: {pop_description} (Spotify popularity score roughly {pop_min}-{pop_max} out of 100).")

    if artist.value.strip():
        similar_names = get_similar_artist_names(artist.value.strip(), n=5)
        if similar_names:
            similar_str = ", ".join(similar_names)
            parts.append(
                f"Artist split: approximately {half} songs by '{artist.value.strip()}', "
                f"and approximately {other_half} songs by similar artists such as: {similar_str}. "
                f"Do NOT include only the specified artist."
            )
        else:
            parts.append(
                f"Artist split: approximately {half} songs by '{artist.value.strip()}', "
                f"and approximately {other_half} songs by artists with a similar sound. "
                f"Do NOT include only the specified artist."
            )
    else:
        parts.append("No specific artist constraint — choose freely.")

    return " ".join(parts)

# --- Parse Claude response into a DataFrame ---
def parse_to_df(result: str):
    match = re.search(r'\[.*\]', result, re.DOTALL)
    if not match:
        return None
    try:
        songs = json.loads(match.group(0))
        df = pd.DataFrame(songs)
        df.columns = [c.replace('_', ' ').title() for c in df.columns]
        df.index = range(1, len(df) + 1)
        return df
    except json.JSONDecodeError:
        return None

# --- Render the table into the single shared output ---
def render_table(df, note=""):
    global current_playlist_df
    current_playlist_df = df
    with playlist_output:
        clear_output(wait=True)
        if note:
            display(widgets.HTML(f"<p style='color:gray;font-size:12px'>{note}</p>"))
        display(df)

# --- Button handlers ---
def on_generate(b):
    global current_playlist_df
    with status_output:
        clear_output()
        print("⏳ Generating playlist...")
        if artist.value.strip():
            print(f"🔍 Looking up similar artists to '{artist.value.strip()}' on Spotify...")

    agent.start_session("custom")
    result = agent.chat(build_prompt(num_songs.value))

    with status_output:
        clear_output()

    df = parse_to_df(result)
    if df is not None:
        render_table(df, note="✅ Playlist generated.")
        finalized_label.value = ""
        # Show the refinement row now that we have a playlist
        refinement_row.layout.display = ''
    else:
        with playlist_output:
            clear_output(wait=True)
            print("⚠️ Could not parse playlist. Raw response:")
            print(result)


def on_refine(b):
    text = refinement_input.value.strip()
    if not text:
        with status_output:
            clear_output()
            print("⚠️ Enter a refinement request first.")
        return

    with status_output:
        clear_output()
        print("⏳ Adjusting...")

    result = agent.chat(text + '\n\nReturn ONLY a JSON array in the same format as before, no other text.')
    refinement_input.value = ""

    with status_output:
        clear_output()

    df = parse_to_df(result)
    if df is not None:
        render_table(df, note="✏️ Playlist updated.")
        finalized_label.value = ""
    else:
        with status_output:
            clear_output()
            print("⚠️ Could not parse refined playlist. Try rephrasing your request.")


def on_done(b):
    if current_playlist_df is not None:
        render_table(current_playlist_df, note="✅ Playlist finalized!")
    finalized_label.value = (
        "<p style='color:green;font-weight:bold;margin-top:8px'>"
        "🎶 Locked in! Copy the table or run agent.export_playlist() to export."
        "</p>"
    )

generate_btn.on_click(on_generate)
refine_btn.on_click(on_refine)
done_btn.on_click(on_done)

# --- Layout ---
display(widgets.VBox([
    widgets.HTML("<h2>🎵 Playlist Generator</h2>"),
    widgets.HTML("<p style='color:gray'>Optional — fill in as many as you like. More = more specific.</p>"),
    widgets.HBox([genre, vibe]),
    widgets.HBox([venue, artist]),
    widgets.HBox([song, album]),
    widgets.HBox([year]),
    widgets.HBox([harmonic_key, key_label]),
    widgets.HBox([bpm_min, bpm_max]),
    widgets.HBox([energy, valence]),
    widgets.HBox([popularity_range, popularity_label]),
    num_songs,
    generate_btn,
    status_output,
    playlist_output,
    refinement_row,      # hidden until playlist is generated
    finalized_label,
]))

  🔧 search_tracks
     └─ {"query": "disco 1975 1976 classic", "limit": 15}
  🔧 search_tracks
     └─ {"query": "disco funk 1977 1978 dance floor", "limit": 15}
  🔧 search_tracks
     └─ {"query": "disco 1979 1980 underground deep cut", "limit": 15}
  🔧 search_tracks
     └─ {"query": "Le Freak Chic", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Got To Be Real Cheryl Lynn", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Never Can Say Goodbye Gloria Gaynor", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Shame Evelyn Champagne King", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Dance Dance Dance Yowsah Chic", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Boogie Wonderland Earth Wind Fire", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Ring My Bell Anita Ward", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Love Hangover Diana Ross", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Heatwave Boogie Nights", "limit": 5}
  🔧 search_tracks
     └─ {"query": "Ten Percent D